# 03. Staged Early Warning Model Benchmark

This notebook evaluates how prediction improves as more timing information becomes available.

Stages:
1. **Stage A:** pre-enrollment variables only
2. **Stage B:** pre-enrollment + first-semester academic performance
3. **Stage C:** pre-enrollment + first- and second-semester academic performance

Main benchmark policy:
- One common input representation for every model: one-hot encoded categorical variables + scaled numeric variables.
- Preprocessing is fitted on training data only and applied unchanged to validation/test data.
- All available benchmark models are trained and evaluated on train, validation, and test sets.

## 1. Imports, Paths, and Output Folders

In [ ]:
from pathlib import Path
import json
import pickle
import time
import warnings

import numpy as np
import pandas as pd

from models.model_registry import MODEL_REGISTRY, available_models, build_model, registry_table
from models.preprocessing import TabularOneHotScaler
from models.evaluation import classification_metrics

warnings.filterwarnings("ignore")

RANDOM_STATE = 123

RESULTS_DIR = Path("results")
PREP_DIR = RESULTS_DIR / "2.Model Processing"
STAGED_DIR = RESULTS_DIR / "3.Staged Early Warning Modeling"
STAGED_CSV_DIR = STAGED_DIR / "csv"
STAGED_PLOTS_DIR = STAGED_DIR / "plots"
STAGED_REPORTS_DIR = STAGED_DIR / "reports"
STAGED_MODELS_DIR = STAGED_DIR / "models"

for folder in [STAGED_CSV_DIR, STAGED_PLOTS_DIR, STAGED_REPORTS_DIR, STAGED_MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MASTER_DATA_PATH = PREP_DIR / "csv" / "modeling_master_dataset_with_splits.csv"
FEATURE_METADATA_PATH = PREP_DIR / "metadata" / "feature_sets_metadata.json"

STAGED_DIR

## 2. Load Prepared Modeling Dataset and Metadata

In [ ]:
df = pd.read_csv(MASTER_DATA_PATH)

with open(FEATURE_METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

stage_feature_sets = metadata["stage_feature_sets"]
categorical_features_all = metadata["feature_types"]["categorical_features"]
numeric_features_all = metadata["feature_types"]["numeric_features"]
target_multiclass = metadata["targets"]["multiclass"]
target_binary = metadata["targets"]["binary_numeric"]
target_binary_label = metadata["targets"]["binary_label"]
split_column = metadata["split_column"]
id_column = metadata["id_column"]

print(df.shape)
display(df[[split_column, target_multiclass]].value_counts().sort_index().rename("n").reset_index())
display(pd.DataFrame([{"stage": k, "n_features": len(v)} for k, v in stage_feature_sets.items()]))

## 3. Model Registry Summary

In [ ]:
registry_df = pd.DataFrame(registry_table())
registry_df.to_csv(STAGED_CSV_DIR / "model_parameter_summary.csv", index=False)

display(registry_df)

available_model_keys = list(MODEL_REGISTRY.keys())
print("Model keys:", available_model_keys)

## 4. Train/Validation/Test Data Helpers

In [ ]:
def get_split_data(data, feature_names, target_col):
    train_df = data[data[split_column] == "train"].copy()
    validation_df = data[data[split_column] == "validation"].copy()
    test_df = data[data[split_column] == "test"].copy()

    X_train_raw = train_df[feature_names]
    X_validation_raw = validation_df[feature_names]
    X_test_raw = test_df[feature_names]

    y_train = train_df[target_col].to_numpy()
    y_validation = validation_df[target_col].to_numpy()
    y_test = test_df[target_col].to_numpy()

    return X_train_raw, X_validation_raw, X_test_raw, y_train, y_validation, y_test


def get_stage_feature_types(stage_features):
    categorical = [feature for feature in stage_features if feature in categorical_features_all]
    numeric = [feature for feature in stage_features if feature in numeric_features_all]
    missing = sorted(set(stage_features) - set(categorical) - set(numeric))
    if missing:
        raise ValueError(f"Features missing type metadata: {missing}")
    return numeric, categorical


def preprocess_stage(data, stage_features, target_col):
    numeric, categorical = get_stage_feature_types(stage_features)
    X_train_raw, X_validation_raw, X_test_raw, y_train, y_validation, y_test = get_split_data(
        data, stage_features, target_col
    )

    preprocessor = TabularOneHotScaler(
        numeric_features=numeric,
        categorical_features=categorical,
    )
    X_train = preprocessor.fit_transform(X_train_raw)
    X_validation = preprocessor.transform(X_validation_raw)
    X_test = preprocessor.transform(X_test_raw)

    return {
        "preprocessor": preprocessor,
        "X_train": X_train,
        "X_validation": X_validation,
        "X_test": X_test,
        "y_train": y_train,
        "y_validation": y_validation,
        "y_test": y_test,
        "numeric_features": numeric,
        "categorical_features": categorical,
    }


stage_shapes = []
for stage_name, stage_features in stage_feature_sets.items():
    prepared = preprocess_stage(df, stage_features, target_multiclass)
    stage_shapes.append({
        "stage": stage_name,
        "raw_features": len(stage_features),
        "encoded_features": prepared["X_train"].shape[1],
        "train_rows": prepared["X_train"].shape[0],
        "validation_rows": prepared["X_validation"].shape[0],
        "test_rows": prepared["X_test"].shape[0],
    })

stage_shapes_df = pd.DataFrame(stage_shapes)
stage_shapes_df.to_csv(STAGED_CSV_DIR / "stage_encoded_feature_shapes.csv", index=False)
stage_shapes_df

## 5. Model Training Helpers

In [ ]:
def model_fit_predict(model_key, prepared):
    spec = MODEL_REGISTRY[model_key]
    model = build_model(model_key)

    X_train = prepared["X_train"].to_numpy(dtype=np.float32)
    X_validation = prepared["X_validation"].to_numpy(dtype=np.float32)
    X_test = prepared["X_test"].to_numpy(dtype=np.float32)
    y_train = prepared["y_train"]
    y_validation = prepared["y_validation"]
    y_test = prepared["y_test"]

    # PyTorch wrapper supports validation data for early stopping.
    if spec.framework == "PyTorch":
        model.fit(X_train, y_train, X_val=X_validation, y_val=y_validation)
    else:
        model.fit(X_train, y_train)

    outputs = {}
    for split_name, X_split, y_split in [
        ("train", X_train, y_train),
        ("validation", X_validation, y_validation),
        ("test", X_test, y_test),
    ]:
        y_pred = model.predict(X_split)
        y_proba = model.predict_proba(X_split) if spec.supports_proba else None
        metrics = classification_metrics(y_split, y_pred, y_proba=y_proba)
        outputs[split_name] = {
            "y_true": y_split,
            "y_pred": y_pred,
            "y_proba": y_proba,
            "metrics": metrics,
        }

    return model, outputs


def save_model_artifact(model, model_key, stage_name):
    model_path = STAGED_MODELS_DIR / f"{stage_name}__{model_key}.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    return model_path

## 6. Run Staged Multiclass Benchmark

In [ ]:
RUN_MODEL_TRAINING = True
SAVE_FITTED_MODELS = False

benchmark_rows = []
skipped_rows = []

if not available_model_keys:
    RUN_MODEL_TRAINING = False

if RUN_MODEL_TRAINING:
    for stage_name, stage_features in stage_feature_sets.items():
        print(f"Preparing {stage_name}...")
        prepared = preprocess_stage(df, stage_features, target_multiclass)

        for model_key in available_model_keys:
            spec = MODEL_REGISTRY[model_key]
            print(f"Training {stage_name} | {spec.display_name}")
            start = time.time()
            try:
                model, outputs = model_fit_predict(model_key, prepared)
                elapsed = time.time() - start

                if SAVE_FITTED_MODELS:
                    model_path = save_model_artifact(model, model_key, stage_name)
                else:
                    model_path = None

                for split_name, split_output in outputs.items():
                    row = {
                        "stage": stage_name,
                        "model_key": model_key,
                        "model": spec.display_name,
                        "framework": spec.framework,
                        "split": split_name,
                        "raw_features": len(stage_features),
                        "encoded_features": prepared["X_train"].shape[1],
                        "fit_seconds": elapsed,
                        "model_path": str(model_path) if model_path else "",
                    }
                    row.update(split_output["metrics"])
                    benchmark_rows.append(row)

            except Exception as exc:
                skipped_rows.append({
                    "stage": stage_name,
                    "model_key": model_key,
                    "model": spec.display_name,
                    "reason": repr(exc),
                })
                print(f"Skipped {stage_name} | {spec.display_name}: {exc}")

benchmark_results = pd.DataFrame(benchmark_rows)
skipped_results = pd.DataFrame(skipped_rows)

benchmark_results.to_csv(STAGED_CSV_DIR / "staged_model_comparison.csv", index=False)
skipped_results.to_csv(STAGED_CSV_DIR / "staged_model_skipped_or_failed.csv", index=False)

display(benchmark_results)
display(skipped_results)

## 7. Select Best Validation Models by Stage

In [ ]:
if benchmark_results.empty:
    best_by_stage = pd.DataFrame(columns=["stage", "model", "macro_f1", "balanced_accuracy", "log_loss"])
else:
    validation_results = benchmark_results[benchmark_results["split"] == "validation"].copy()
    best_by_stage = (
        validation_results.sort_values(
            ["stage", "macro_f1", "balanced_accuracy"],
            ascending=[True, False, False],
        )
        .groupby("stage", as_index=False)
        .head(1)
    )

best_by_stage.to_csv(STAGED_CSV_DIR / "best_validation_model_by_stage.csv", index=False)
best_by_stage

## 8. Create Simple CSV Report Tables

In [ ]:
if benchmark_results.empty:
    validation_pivot = pd.DataFrame()
    test_pivot = pd.DataFrame()
else:
    validation_pivot = benchmark_results[benchmark_results["split"] == "validation"].pivot_table(
        index=["model"],
        columns="stage",
        values=["macro_f1", "balanced_accuracy", "log_loss"],
        aggfunc="first",
    )
    test_pivot = benchmark_results[benchmark_results["split"] == "test"].pivot_table(
        index=["model"],
        columns="stage",
        values=["macro_f1", "balanced_accuracy", "log_loss"],
        aggfunc="first",
    )

validation_pivot.to_csv(STAGED_CSV_DIR / "validation_stage_metric_pivot.csv")
test_pivot.to_csv(STAGED_CSV_DIR / "test_stage_metric_pivot.csv")

display(validation_pivot)
display(test_pivot)

## 9. Write Run Summary Report

In [ ]:
summary_lines = []
summary_lines.append("# Staged Early Warning Model Benchmark Run Summary\n")
summary_lines.append("## Models\n")
for _, row in registry_df.iterrows():
    summary_lines.append(f"- {row['model']}")

summary_lines.append("\n## Stage Feature Shapes\n")
for _, row in stage_shapes_df.iterrows():
    summary_lines.append(
        f"- {row['stage']}: {row['raw_features']} raw features, {row['encoded_features']} encoded features"
    )

summary_lines.append("\n## Run Status\n")
if benchmark_results.empty:
    summary_lines.append("No models were trained. Check package installation and rerun this notebook.")
else:
    summary_lines.append(f"Trained result rows: {len(benchmark_results)}")
    summary_lines.append("Best validation models are saved in `csv/best_validation_model_by_stage.csv`.")

summary_path = STAGED_REPORTS_DIR / "staged_early_warning_run_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
summary_path

## 10. Next Step

After this notebook successfully runs, the next notebook should be:

`5_Variable_Block_Ablation_Study.ipynb`

That notebook should use the same preprocessing logic and the best-performing model family from this staged benchmark.